In [1]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

In [2]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")

In [3]:
train_transaction = pd.read_csv("../data/train_transaction.csv")
train_identity = pd.read_csv("../data/train_identity.csv")

df = train_transaction.merge(train_identity, on="TransactionID", how="left")

In [4]:
df = df.drop(columns=["TransactionID"], errors="ignore")

y = df["isFraud"]
X = df.drop(columns=["isFraud"])

In [5]:
X = X.sample(150000, random_state=42)
y = y.loc[X.index]

Cleaning

In [6]:
mlflow.set_experiment("LogisticRegression_Training")

with mlflow.start_run(run_name="LogReg_Cleaning"):
    null_thresh = 0.8
    cols_to_drop = [c for c in X.columns if X[c].isnull().mean() > null_thresh]
    X = X.drop(columns=cols_to_drop)

    mlflow.log_param("null_threshhold", null_thresh)
    mlflow.log_param("cols_dropped", len(cols_to_drop))
    mlflow.log_metric("cols_remaining", X.shape[1])

    print(f"Dropped {len(cols_to_drop)} high-null columns. Remaining: {X.shape[1]}")

Dropped 74 high-null columns. Remaining: 358
🏃 View run LogReg_Cleaning at: http://127.0.0.1:5000/#/experiments/1/runs/b09def1948ae41c49e71c0bf8adbd4fe
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Feature Engineering

In [7]:
with mlflow.start_run(run_name="LogReg_FeatureEngineering"):
    X["TransactionAmt_log"] = np.log1p(X["TransactionAmt"])
    X["hour_of_day"] = (X["TransactionDT"] / 3600).astype(int) % 24
    X["null_count"] = X.isnull().sum(axis=1)

    new_features = ["TransactionAmt_log", "hour_of_day", "null_count"]
    mlflow.log_param("new_features", new_features)
    mlflow.log_metric("total_cols_after_eng", X.shape[1])
    print("Feature Engineering Done: ", new_features)

Feature Engineering Done:  ['TransactionAmt_log', 'hour_of_day', 'null_count']
🏃 View run LogReg_FeatureEngineering at: http://127.0.0.1:5000/#/experiments/1/runs/0a49c12dde244e37a0aed6a27a5e7c6b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Feature Selection

In [8]:
with mlflow.start_run(run_name="LogReg_FeatureSelection"):
    num_only = X.select_dtypes(include=["int64", "float64"]).fillna(0)
    corr_matrix = num_only.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    drop_corr = [col for col in upper.columns if any(upper[col] > 0.95)]
    X = X.drop(columns=drop_corr, errors="ignore")

    mlflow.log_param("corr_threshold", 0.95)
    mlflow.log_metric("cols_dropped_corr", len(drop_corr))
    mlflow.log_metric("cols_remaining", X.shape[1])
    print(f"Dropped {len(drop_corr)} correlated columns. Remaining: {X.shape[1]}")

Dropped 109 correlated columns. Remaining: 252
🏃 View run LogReg_FeatureSelection at: http://127.0.0.1:5000/#/experiments/1/runs/d5e85b2a289b4b1591691683ca1cd87b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [9]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(max_iter=200, solver="saga", n_jobs=-1))
])

In [10]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

Training

In [11]:
with mlflow.start_run(run_name="LogReg_Training"):
    pipeline.fit(X_train, y_train)

    preds = pipeline.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, preds)

    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("solver", "saga")
    mlflow.log_param("max_iter", 200)
    mlflow.log_param("sample_size", 150000)
    mlflow.log_metric("auc", auc)

    mlflow.sklearn.log_model(pipeline, "pipeline_model", registered_model_name="LogisticRegression_FraudDetection")

    print("AUC:", auc)

c:\Users\andriam\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\andriam\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
2026/05/03 20:00:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 20:00:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persist

AUC: 0.838926717844377
🏃 View run LogReg_Training at: http://127.0.0.1:5000/#/experiments/1/runs/78ed4312b4734df693888293b7ebe57e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
